# Context-aware chord-refinement Transformer, from scratch

This notebook uses the immutable **v6 multi-bar dataset** and the actual modules in `src/model/`. One token is one beat; one output is a detailed chord-class distribution. It demonstrates architecture correctness and **one optimizer step**, not full training or musical accuracy.

```text
Basic chord ----------> Chord encoder ------------------+
Pitch + timing -------> Real-note masked mean encoder --+
Key/mode + Roman/function -> Harmonic encoder -----------+-> concatenate -> Linear(128)
Genre + beat/bar timing -> Context encoder --------------+   -> norm/dropout -> sinusoidal positions
                                                            -> 2 bidirectional pre-norm blocks
                                                            -> final norm -> Linear(C) -> beat logits
```

Select the repository `.venv` Python kernel in VS Code. Run all cells from the notebook directory or repository root. CPU is the default; CUDA is optional.

## 1. Load the saved vocabulary contract and a real batch

NPZ names have `inputs__`/`targets__` prefixes; `SavedSequenceDataset` returns nested dictionaries **without** these prefixes. No vocabulary is fitted here. IDs are categorical integers, masks are booleans, timing is float32, and genre is sequence-level `[B,G]`.

Melody extraction source IDs remain audit data in the saved dataset; they are excluded from model inputs.

In [1]:
from pathlib import Path
import sys
import torch
from IPython.display import display
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'experiments/rahul/src').is_dir()), None)
if root is None:
    raise RuntimeError('Run inside the vocal-harmonization repository')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from experiments.rahul.src.model import DatasetSpec, ModelConfig, ChordRefinementTransformer, masked_cross_entropy
from experiments.rahul.src.transformer_dataset import SavedSequenceDataset, make_dataloader, collate_sequences
from experiments.rahul.src.inspect_transformer import mixed_length_examples
folder = root/'experiments/rahul/outputs/model_preprocessing/final_all_tracks_quality_v6_multibar'
device = torch.device('cpu')  # Optional: torch.device('cuda') if torch.cuda.is_available()
torch.set_num_threads(1)
torch.manual_seed(42)
spec = DatasetSpec.from_folder(folder)
settings = ModelConfig()
model = ChordRefinementTransformer(spec, settings).to(device).eval()
dataset = SavedSequenceDataset(folder/'train_sequences.json')
batch = next(iter(make_dataloader(dataset, batch_size=4, shuffle=False)))
inputs = {k:v.to(device) for k,v in batch['inputs'].items() if k != 'melody_source_ids'}
targets = {k:v.to(device) for k,v in batch['targets'].items()}
display({'sequences':len(dataset), 'target_classes':spec.num_target_classes,
         'genre_dim':spec.genre_dim, 'note_slots':spec.note_capacity,
         'parameters':sum(p.numel() for p in model.parameters())})
display({k:{'shape':list(v.shape), 'dtype':str(v.dtype)} for k,v in inputs.items()})

{'sequences': 12718,
 'target_classes': 402,
 'genre_dim': 63,
 'note_slots': 32,
 'parameters': 349010}

{'basic_chord_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'basic_root_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'basic_quality_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'previous_basic_chord_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'next_basic_chord_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'key_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'mode_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'key_confidence_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'roman_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'harmonic_function_ids': {'shape': [4, 16], 'dtype': 'torch.int64'},
 'melody_pitch_ids': {'shape': [4, 16, 32], 'dtype': 'torch.int64'},
 'melody_pitch_class_ids': {'shape': [4, 16, 32], 'dtype': 'torch.int64'},
 'melody_pitch_values': {'shape': [4, 16, 32], 'dtype': 'torch.float32'},
 'melody_timing': {'shape': [4, 16, 32, 3], 'dtype': 'torch.float32'},
 'melody_note_mask': {'shape': [4, 16, 32], 'dtype': 'torch.bool'},


## 2. Chord encoder

The default uses only `basic_chord_ids`: a learned lookup `[B,L] ? [B,L,32]`. Root/quality and previous/next chord embeddings are opt-in. This avoids making redundant chord features mandatory.

In [2]:
chord = model.encoders['chord'](inputs)
print('Chord:', tuple(chord.shape))
print('First beat embedding:', chord[0,0,:8].detach().cpu())

Chord: (4, 16, 32)
First beat embedding: tensor([ 0.6462,  0.9474, -0.9643, -0.0554,  0.6487, -1.5245, -0.4629, -1.6246])


## 3. Melody-note encoding and masked mean pooling

Each note combines a categorical pitch embedding with a projection of `[duration/beat_duration, relative_onset/beat_duration, overlap/beat_duration]`. Layer normalization and GELU form the note representation.

For real-note mask $m_n$, pooling is $h=\sum_n m_n h_n / \max(1,\sum_n m_n)$. Only **real notes** contribute; the silent-beat dummy attention slot does not. A learned silence indicator is added. Negative onsets and duration ratios above one can be valid sustained notes.

In [3]:
melody_encoder = model.encoders['melody']
notes = melody_encoder.encode_notes(inputs)
melody = melody_encoder(inputs)
print('Notes:',tuple(notes.shape),'Pooled:',tuple(melody.shape))
print('Note counts:',inputs['melody_note_mask'][0].sum(-1).cpu())
print('First beat timing:', inputs['melody_timing'][0,0][inputs['melody_note_mask'][0,0]].cpu())
silent_inputs = dict(inputs)
silent_inputs['melody_note_mask'] = torch.zeros_like(inputs['melody_note_mask'])
silent_inputs['melody_silent'] = torch.ones_like(inputs['melody_silent'])
print('All-silent encoding finite:',bool(torch.isfinite(melody_encoder(silent_inputs)).all()))

Notes: (4, 16, 32, 32) Pooled: (4, 16, 32)
Note counts: tensor([1, 2, 3, 2, 2, 2, 3, 1, 1, 3, 1, 2, 1, 3, 1, 2])
First beat timing: tensor([[1.1522, 0.0000, 1.0000]])
All-silent encoding finite: True


## 4. Multi-hot genre projection and temporal context

Genre is projected with a linear layer `[B,G] ? [B,32]`, then broadcast over beats. Multiple active tags are accepted.

Beat position and relative bar number are saved **numerical float32 values**, not vocabulary IDs. They use `log1p` projections, along with `log1p(duration_seconds)` and the two known-timing flags. Unknown bar/position values are zeroed before projection. Absolute bar number and beat index are not model features.

In [4]:
context_encoder = model.encoders['genre_temporal']
genre = context_encoder.genre(inputs['genre_features'])
temporal_values = context_encoder.temporal_values(inputs)
genre_temporal = context_encoder(inputs)
print('Active genre coordinates:',torch.nonzero(inputs['genre_features'][0]).flatten().cpu().tolist())
print('Genre:',tuple(genre.shape),'Numerical timing:',tuple(temporal_values.shape))
print('Known timing [bar,position]:',inputs['beat_timing_known_mask'][0,:4].cpu())
print('Context:',tuple(genre_temporal.shape))

Active genre coordinates: [14, 45]
Genre: (4, 32) Numerical timing: (4, 16, 5)
Known timing [bar,position]: tensor([[True, True],
        [True, True],
        [True, True],
        [True, True]])
Context: (4, 16, 32)


## 5. Harmonic context

Separate learned embeddings encode key, mode, Roman numeral and harmonic function. They are concatenated and projected to 32 dimensions. Source preprocessing computes Roman/function from **basic chord + melody-estimated key/mode**; source reference Roman labels are audit-only. Basic chords themselves remain synthetic reference-derived inputs, which is a research limitation.

In [5]:
harmonic_encoder = model.encoders['harmonic']
display({k:list(embedding(inputs[k]).shape) for k,embedding in harmonic_encoder.embeddings.items()})
harmonic = harmonic_encoder(inputs)
print('Harmonic context:',tuple(harmonic.shape))

{'key_ids': [4, 16, 32],
 'mode_ids': [4, 16, 32],
 'roman_ids': [4, 16, 32],
 'harmonic_function_ids': [4, 16, 32]}

Harmonic context: (4, 16, 32)


## 6. Feature fusion

Four 32-dimensional representations are concatenated into 128 features, followed by a learned projection to `d_model=128`, layer normalization and dropout. Disabled groups are omitted from the concatenation for ablations.

In [6]:
features = model.encode_features(inputs)
concatenated = torch.cat(list(features.values()),dim=-1)
projected = model.fusion(concatenated)
print('Concatenated:',tuple(concatenated.shape),'Projected:',tuple(projected.shape))
display({name:list(value.shape) for name,value in features.items()})

Concatenated: (4, 16, 128) Projected: (4, 16, 128)


{'chord': [4, 16, 32],
 'melody': [4, 16, 32],
 'harmonic': [4, 16, 32],
 'genre_temporal': [4, 16, 32]}

## 7. Explicit sinusoidal sequence positions

$PE(p,2i)=\sin(p/10000^{2i/d})$, $PE(p,2i+1)=\cos(p/10000^{2i/d})$.

Positions are generated at the current sequence length; no fixed 16-beat assumption or learned maximum bar index is needed. Padded beat representations are zeroed after addition.

In [7]:
fused = model.fuse_features(features, inputs['beat_attention_mask'])
position_only = model.position(torch.zeros_like(projected))
print('Position vectors (first two, first six coordinates):',position_only[0,:2,:6].cpu())
print('Fused:',tuple(fused.shape))

Position vectors (first two, first six coordinates): tensor([[0.0000, 1.0000, 0.0000, 1.0000, 0.0000, 1.0000],
        [0.8415, 0.5403, 0.7617, 0.6479, 0.6816, 0.7318]])
Fused: (4, 16, 128)


## 8. Query, key and value projections

Each block uses pre-normalization. Independent learned linear projections produce Q, K and V, then reshape `[B,L,128] ? [B,4,L,32]`. Four heads each have dimension $d_k=32$.

In [8]:
block = model.layers[0]
normalized = block.norm1(fused)
query, key, value = block.attention.project_qkv(normalized)
print('Q:',tuple(query.shape),'K:',tuple(key.shape),'V:',tuple(value.shape))

Q: (4, 4, 16, 32) K: (4, 4, 16, 32) V: (4, 4, 16, 32)


## 9. Scaled dot-product attention

$A=\mathrm{softmax}(QK^T/\sqrt{d_k})$ and $O=AV$. Invalid **keys** receive negative infinity before softmax. There is no causal triangle: each real beat can attend to earlier and later beats. Padded query rows are zeroed. Completely padded examples are handled with a zero distribution.

In [9]:
import math
scores = query @ key.transpose(-2,-1) / math.sqrt(block.attention.head_dim)
key_padding_mask = ~inputs['beat_attention_mask']
masked_scores = scores.masked_fill(key_padding_mask[:,None,None,:], -torch.inf)
attention_output, weights = block.attention(normalized, inputs['beat_attention_mask'], return_attention=True)
print('Scores:',tuple(scores.shape),'Weights:',tuple(weights.shape))
print('First query attention over first six keys:', weights[0,0,0,:6].detach().cpu())
print('First beat can attend to last real beat:',float(weights[0,0,0,-1].detach()))

Scores: (4, 4, 16, 16) Weights: (4, 4, 16, 16)
First query attention over first six keys: tensor([0.0838, 0.0736, 0.0588, 0.0659, 0.0703, 0.0697])
First beat can attend to last real beat: 0.049476612359285355


## 10. Multi-head aggregation

Each head aggregates values using its attention weights. The heads concatenate back to `[B,L,128]`, followed by an output projection. The implementation uses ordinary matrix operations and linear layers, not `nn.MultiheadAttention` or a ready-made Transformer.

In [10]:
head_values = weights @ value  # eval mode: attention dropout disabled
joined = head_values.transpose(1,2).contiguous().reshape_as(fused)
print('Per-head values:',tuple(head_values.shape),'Joined:',tuple(joined.shape))
print('Attention output:',tuple(attention_output.shape))

Per-head values: (4, 4, 16, 32) Joined: (4, 16, 128)
Attention output: (4, 16, 128)


## 11. Pre-norm residual block and feed-forward network

$u=x+\mathrm{Dropout}(\mathrm{Attention}(\mathrm{LN}(x)))$

$y=u+\mathrm{Dropout}(W_2\mathrm{Dropout}(\mathrm{GELU}(W_1\mathrm{LN}(u))))$.

The feed-forward width is 256. Two blocks are stacked, followed by a final layer normalization. Padding is zeroed after every block.

In [11]:
residual = fused + block.dropout(attention_output)
feed_forward = block.feed_forward(block.norm2(residual))
block_output = block(fused, inputs['beat_attention_mask'])
print('Residual:',tuple(residual.shape),'FFN:',tuple(feed_forward.shape),'Block:',tuple(block_output.shape))

Residual: (4, 16, 128) FFN: (4, 16, 128) Block: (4, 16, 128)


## 12. Beat-level classification and untrained top-three candidates

The final `Linear(128,C)` produces raw logits, with $C$ read from the saved target vocabulary. Cross-entropy consumes logits directly. For readable inference, `predict_topk` applies softmax over musical classes, excluding PAD/UNK, and returns empty lists for padded beats.

These predictions are from random initialization and have no musical validity.

In [12]:
with torch.no_grad():
    logits, details = model(inputs, return_details=True)
print('Logits:',tuple(logits.shape),'First eight logits:',logits[0,0,:8].cpu())
print('Reference:',spec.target_labels[int(targets['target_chord_ids'][0,0])])
display(model.predict_topk(inputs)[0][0])

Logits: (4, 16, 402) First eight logits: 

tensor([ 0.1746, -0.5242, -0.7013, -0.0256,  0.8778,  0.2997, -0.2990, -0.0197])
Reference: G:maj


[{'id': 161, 'label': 'C:min7/4', 'probability': 0.009038738906383514},
 {'id': 159, 'label': 'C:min/b7', 'probability': 0.008951747789978981},
 {'id': 289, 'label': 'F#:maj/7', 'probability': 0.008669815957546234}]

## 13. Masked cross-entropy and real variable-length batching

Only `target_loss_mask & beat_attention_mask` contributes. Ambiguous real beats remain attention context. Ignored targets are -100; an empty supervised batch returns a differentiable zero.

The examples below are located from actual shard offsets, not made by slicing or inventing source beats.

In [13]:
loss = masked_cross_entropy(logits, targets, inputs['beat_attention_mask'])
print('Initial eval loss:',float(loss))
print('Valid targets:',int((targets['target_loss_mask'] & inputs['beat_attention_mask']).sum()))
mixed = collate_sequences(mixed_length_examples(folder))
mixed_inputs = {k:v.to(device) for k,v in mixed['inputs'].items() if k != 'melody_source_ids'}
with torch.no_grad():
    mixed_logits = model(mixed_inputs)
print('Real lengths:',mixed_inputs['beat_attention_mask'].sum(-1).cpu().tolist())
print('Mixed logits:',tuple(mixed_logits.shape),'Padded beats:',int((~mixed_inputs['beat_attention_mask']).sum()))
print('Beat mask:',mixed_inputs['beat_attention_mask'].cpu())
assert (mixed_logits[~mixed_inputs['beat_attention_mask']] == 0).all()

Initial eval loss: 6.297266483306885
Valid targets: 64


Real lengths: [12, 16, 20]
Mixed logits: (3, 20, 402) Padded beats: 12
Beat mask: tensor([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True, False, False, False, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True]])


## 14. Backpropagation and exactly one optimizer step

This is a wiring check, not a training experiment. All instantiated trainable parameters must receive finite gradients, and an optimizer step must change weights. No accuracy, validation score or trained checkpoint is produced.

In [14]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)
before = {name:p.detach().clone() for name,p in model.named_parameters()}
optimizer.zero_grad(set_to_none=True)
step_loss = masked_cross_entropy(model(inputs),targets,inputs['beat_attention_mask'])
step_loss.backward()
gradient_norms = {name:float(p.grad.norm()) if p.grad is not None else None for name,p in model.named_parameters()}
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
optimizer.step()
changed = [name for name,p in model.named_parameters() if not torch.equal(before[name],p)]
assert changed
print('Dropout-enabled debug loss:',float(step_loss.detach()))
print('Finite gradients for every parameter; changed parameter tensors:',len(changed))
display(gradient_norms)
model.eval();

Dropout-enabled debug loss: 6.313750267028809
Finite gradients for every parameter; changed parameter tensors: 59


{'encoders.chord.embeddings.basic_chord_ids.weight': 0.10115950554609299,
 'encoders.melody.pitch.weight': 0.05717834457755089,
 'encoders.melody.timing.weight': 0.08759178221225739,
 'encoders.melody.timing.bias': 0.08125229924917221,
 'encoders.melody.note_norm.weight': 0.04461904242634773,
 'encoders.melody.note_norm.bias': 0.0906735360622406,
 'encoders.melody.silence.weight': 0.0906735360622406,
 'encoders.harmonic.embeddings.key_ids.weight': 0.02954641543328762,
 'encoders.harmonic.embeddings.mode_ids.weight': 0.02559269592165947,
 'encoders.harmonic.embeddings.roman_ids.weight': 0.027771230787038803,
 'encoders.harmonic.embeddings.harmonic_function_ids.weight': 0.02716670371592045,
 'encoders.harmonic.projection.weight': 1.1028403043746948,
 'encoders.harmonic.projection.bias': 0.10208781808614731,
 'encoders.genre_temporal.genre.weight': 0.3011091351509094,
 'encoders.genre_temporal.genre.bias': 0.2129163146018982,
 'encoders.genre_temporal.temporal.weight': 0.4252401292324066,

## Reuse, ablations and limitations

`ModelConfig.experiment('baseline'/'A'/'B'/'C'/'full')` selects chord-only through full-context variants. Root/quality, neighbor chords and extra melody representations are opt-in settings. All variants share the same attention implementation.

Detailed chords are automatic-recognition pseudo-labels. Basic chords are synthetic simplifications of those same references. This model initially learns refinement under that construction. Independent Member 2 basic-chord predictions are needed for realistic integrated evaluation. Theory-derived context rules require ablations. A successful forward/backward pass does not validate musical quality.

The dataset and vocabularies remain unchanged. Run the architecture tests with:

```powershell
.\.venv\Scripts\python.exe -m unittest experiments.rahul.tests.test_chord_refinement_transformer -q
```